In [1]:
!pip -q install --no-cache-dir "numpy==2.0.2" "pandas==2.2.2" "scikit-learn==1.6.1" \
"tqdm==4.67.1" "safetensors==0.5.3" \
"huggingface-hub==0.30.2" "tokenizers==0.21.1" "transformers==4.51.3" \
"accelerate==1.6.0" "bitsandbytes==0.48.2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 224.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 343.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 359.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 388.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 264.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 243.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 351.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 254.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-c

# End-to-end evaluation

Pipeline được đánh giá:

```text
Claim → retrieval Top-5 evidence → LLM verifier → final label
```

Các cấu hình chính:

- BM25 → Qwen2.5-3B.
- Dense AITeamVN → Qwen2.5-3B.
- Hybrid → Qwen2.5-3B.
- Hybrid + reranker → Qwen2.5-3B.
- Hybrid + reranker → Qwen2.5-7B 8-bit.

Accuracy và Macro-F1 được tính trên toàn bộ ba nhãn. Evidence Recall@5 và FEVER-style score được tính trên các claim `SUPPORTED` và `REFUTED`.


## Config


In [2]:
import re, gc
import json
import shutil
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Change only this variable when switching from dev to test.
RUN_TAG = "test"  # "dev" or "test"

DATASET_PATHS = {
    "dev": "/kaggle/input/datasets/lanthanhvi/datasett/dev_dataset.jsonl",
    "test": "/kaggle/input/datasets/lanthanhvi/datasett/test_dataset.jsonl",
}

# Replace the paths below with the exact retrieval output files generated by your retrieval notebook.

RETRIEVAL_PATHS = {
    "dev": {
        "bm25": "/kaggle/input/datasets/lanthanhvi/retrieval-output-dev/bm25_top20_all_labels_dev.jsonl",
        "dense": "/kaggle/input/datasets/lanthanhvi/retrieval-output-dev/dense_top20_all_labels_dev.jsonl",
        "hybrid": "/kaggle/input/datasets/lanthanhvi/retrieval-output-dev/hybrid_alpha05_top20_all_labels_dev.jsonl",
        "hybrid_reranker": "/kaggle/input/datasets/lanthanhvi/retrieval-output-dev/hybrid_reranker_alpha05_top10_all_labels_dev.jsonl",
    },
    "test": {
        "bm25": "/kaggle/input/datasets/lanthanhvi/retrieval-output-test/bm25_top20_all_labels_test.jsonl",
        "dense": "/kaggle/input/datasets/lanthanhvi/retrieval-output-test/dense_top20_all_labels_test.jsonl",
        "hybrid": "/kaggle/input/datasets/lanthanhvi/retrieval-output-test/hybrid_alpha05_top20_all_labels_test.jsonl",
        "hybrid_reranker": "/kaggle/input/datasets/lanthanhvi/retrieval-output-test/hybrid_reranker_alpha05_top10_all_labels_test.jsonl",
    },
}

CORPUS_PATH = "/kaggle/input/datasets/lanthanhvi/corpus/full_DVSK_data_aiteamvn_256.json"
OUTPUT_DIR = Path("/kaggle/working/datasets/lanthanhvi/end_to_end_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = {
    "qwen25_3b_full": {"name": "Qwen/Qwen2.5-3B-Instruct", "quantization": None},
    "qwen25_7b_8bit": {"name": "Qwen/Qwen2.5-7B-Instruct", "quantization": "8bit"},
}

# Qwen 3B is used to compare all retrieval methods.
# Qwen 7B is evaluated only with the strongest retrieval pipeline.
EXPERIMENTS = [
    {"tag": "bm25_qwen25_3b_full", "retrieval_method": "bm25", "model_tag": "qwen25_3b_full"},
    {"tag": "dense_qwen25_3b_full", "retrieval_method": "dense", "model_tag": "qwen25_3b_full"},
    {"tag": "hybrid_qwen25_3b_full", "retrieval_method": "hybrid", "model_tag": "qwen25_3b_full"},
    {"tag": "hybrid_reranker_qwen25_3b_full", "retrieval_method": "hybrid_reranker", "model_tag": "qwen25_3b_full"},
    {"tag": "hybrid_reranker_qwen25_7b_8bit", "retrieval_method": "hybrid_reranker", "model_tag": "qwen25_7b_8bit"},
]

LABELS = ["SUPPORTED", "REFUTED", "NOT_ENOUGH_EVIDENCE"]
VERIFIABLE_LABELS = ["SUPPORTED", "REFUTED"]
EXTENDED_LABELS = LABELS + ["PARSE_ERROR"]

MAX_EVIDENCE_ITEMS = 5
MAX_SAMPLES = None  # Set to None for the official full run.
FORCE_RERUN = True

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
secret_label = "HF_ViHiFact"

HF_TOKEN = UserSecretsClient().get_secret(secret_label)
HF_REPO_ID = "ViLannnn/ViHisFact"

api = HfApi() 

## Đọc dữ liệu


In [3]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def read_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


dataset = read_jsonl(DATASET_PATHS[RUN_TAG])
if MAX_SAMPLES is not None:
    dataset = dataset[:MAX_SAMPLES]

corpus = read_json_list(CORPUS_PATH)
corpus_by_id = {str(row["chunk_id"]): row for row in corpus}

retrieval_rows_by_method = {}
retrieved_maps = {}

for method, path in RETRIEVAL_PATHS[RUN_TAG].items():
    rows = read_jsonl(path)
    retrieval_rows_by_method[method] = rows
    retrieved_maps[method] = {str(row["claim_id"]): row for row in rows}

print("RUN_TAG:", RUN_TAG)
print("Dataset size:", len(dataset))
print("Corpus size:", len(corpus_by_id))

for method, retrieved_map in retrieved_maps.items():
    print(f"{method} retrieval size:", len(retrieved_map))

print("\nLabel distribution:")
print(pd.Series([row["label"] for row in dataset]).value_counts())

print("\nClaim type distribution:")
print(pd.Series([row["claim_type"] for row in dataset]).value_counts())

# Warn before a test run if test paths still point to dev retrieval files.
if RUN_TAG == "test":
    duplicated_methods = [
        method for method in RETRIEVAL_PATHS["test"]
        if RETRIEVAL_PATHS["test"][method] == RETRIEVAL_PATHS["dev"][method]
    ]
    if duplicated_methods:
        print("\nWARNING: Test retrieval paths still duplicate dev paths:", duplicated_methods)


RUN_TAG: test
Dataset size: 1050
Corpus size: 5343
bm25 retrieval size: 1050
dense retrieval size: 1050
hybrid retrieval size: 1050
hybrid_reranker retrieval size: 1050

Label distribution:
SUPPORTED              350
NOT_ENOUGH_EVIDENCE    350
REFUTED                350
Name: count, dtype: int64

Claim type distribution:
MILITARY_DIPLOMACY          357
PERSON_ROLE                 254
POLITICAL_ADMINISTRATIVE    159
TIME_EVENT                  151
GENEALOGY_SUCCESSION         81
NATURAL_OMEN_DISASTER        48
Name: count, dtype: int64


## Chuẩn bị evidence và prompt


In [4]:
def get_raw_text_from_chunk_id(chunk_id):
    chunk = corpus_by_id.get(str(chunk_id))
    if chunk is None:
        raise KeyError(f"Chunk ID was not found in the corpus: {chunk_id}")

    raw_text = str(chunk.get("raw_text", "")).strip()
    if not raw_text:
        raise ValueError(f"Chunk has no valid raw_text: {chunk_id}")

    return raw_text


def get_retrieval_info(item, retrieved_map):
    claim_id = str(item["claim_id"])
    retrieval_row = retrieved_map[claim_id]
    evidences = retrieval_row["retrieved_evidence"][:MAX_EVIDENCE_ITEMS]

    # Gold evidence always comes from the benchmark dataset, not the retrieval output.
    gold_evidence_ids = {str(chunk_id) for chunk_id in item["evidence_ids"]}
    retrieved_chunk_ids = [str(evidence["chunk_id"]) for evidence in evidences]

    gold_rank = None
    for index, evidence in enumerate(evidences, start=1):
        chunk_id = str(evidence["chunk_id"])
        rank = int(evidence.get("rank", index))
        if chunk_id in gold_evidence_ids:
            gold_rank = rank
            break

    return {
        "gold_evidence_ids": sorted(gold_evidence_ids),
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "retrieval_hit_at_5": gold_rank is not None,
        "gold_rank": gold_rank,
    }


def format_retrieved_evidence(item, retrieved_map):
    claim_id = str(item["claim_id"])
    evidences = retrieved_map[claim_id]["retrieved_evidence"][:MAX_EVIDENCE_ITEMS]
    blocks = []

    for index, evidence in enumerate(evidences, start=1):
        chunk_id = str(evidence["chunk_id"])
        rank = int(evidence.get("rank", index))
        raw_text = get_raw_text_from_chunk_id(chunk_id)
        blocks.append(f"Evidence {rank}:\n{raw_text}")

    return "\n\n".join(blocks)


def build_prompt(item, retrieved_map):
    claim = str(item["claim"]).strip()
    evidence_text = format_retrieved_evidence(item, retrieved_map)

    return f"""Bạn là hệ thống kiểm chứng phát biểu lịch sử.
Hãy xác định quan hệ giữa CLAIM và EVIDENCE chỉ dựa trên bằng chứng được cung cấp. Không sử dụng kiến thức bên ngoài và không suy diễn vượt quá nội dung bằng chứng.

Nhãn:
- SUPPORTED: EVIDENCE cung cấp đủ thông tin để xác nhận tất cả nội dung trọng tâm của CLAIM.
- REFUTED: EVIDENCE trực tiếp mâu thuẫn với ít nhất một nội dung trọng tâm có vai trò quyết định trong CLAIM.
- NOT_ENOUGH_EVIDENCE: EVIDENCE không cung cấp đủ thông tin để xác nhận hoặc bác bỏ CLAIM. Việc EVIDENCE không đề cập đến một chi tiết không đồng nghĩa với việc chi tiết đó bị phản bác.

Quy tắc:
- Kiểm tra các yếu tố trọng tâm như nhân vật, thời gian, địa điểm, chức vụ, quan hệ, hành động và kết quả.
- Chỉ chọn SUPPORTED khi tất cả nội dung trọng tâm được xác nhận.
- Chọn REFUTED khi có mâu thuẫn trực tiếp và rõ ràng.
- Chọn NOT_ENOUGH_EVIDENCE khi bằng chứng thiếu thông tin cần thiết hoặc chỉ liên quan chung đến chủ đề.
- Không chọn SUPPORTED chỉ vì CLAIM và EVIDENCE có nhiều từ hoặc thực thể giống nhau.
- Chỉ trả về đúng một JSON hợp lệ, không thêm Markdown hoặc văn bản bên ngoài JSON.

CLAIM:
{claim}

EVIDENCE:
{evidence_text}

Định dạng đầu ra:
{{"label":"SUPPORTED hoặc REFUTED hoặc NOT_ENOUGH_EVIDENCE","explanation":"Giải thích ngắn gọn trong tối đa hai câu."}}"""

## Parse output và gọi LLM


In [5]:
def parse_verification_output(raw_text):
    text = (raw_text or "").strip()
    candidates = []

    # 1. JSON trong code block
    fence = re.search(
        r"```(?:json)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )
    if fence:
        candidates.append(fence.group(1).strip())

    # 2. Toàn bộ text có thể là JSON
    candidates.append(text)

    # 3. JSON object nằm trong text
    json_obj = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if json_obj:
        candidates.append(json_obj.group(0).strip())

    last_label = ""
    last_explanation = ""

    for cand in candidates:
        try:
            # Sửa lỗi model trả thừa một dấu " ngay trước dấu }
            cand = re.sub(r'""\s*}', '"}', cand)

            obj = json.loads(cand)

            label = str(obj.get("label", "")).strip().upper()
            explanation = str(obj.get("explanation", "")).strip()

            last_label = label
            last_explanation = explanation

            if label not in LABELS:
                continue

            if not explanation:
                return label, "", "missing_explanation"

            return label, explanation, None

        except Exception:
            continue

    return (
        "PARSE_ERROR",
        last_explanation,
        f"parse_error_or_invalid_label: {last_label}"
    )


def generate_one(tokenizer, model, prompt):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    input_device = model.get_input_embeddings().weight.device
    inputs = inputs.to(input_device)
    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            temperature=None,    
            top_p=None,         
            top_k=None,    
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0, input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


## Tải tokenizer và model


In [6]:
def get_quant_config(quantization):
    if quantization is None:
        return None
    if quantization == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    raise ValueError(f"Unsupported quantization: {quantization}")


def load_llm(model_name, quantization):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    load_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16,
        "low_cpu_mem_usage": True,
    }

    quant_config = get_quant_config(quantization)
    if quant_config is not None:
        load_kwargs["quantization_config"] = quant_config

    model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
    model.eval()
    return tokenizer, model


## Chạy dự đoán end-to-end


In [7]:
def classify_end_to_end_result(gold_label, label_correct, retrieval_hit_at_5):
    # NEI does not use the annotated source chunk as direct supporting or refuting evidence.
    if gold_label == "NOT_ENOUGH_EVIDENCE":
        return "nei_label_correct" if label_correct else "nei_label_incorrect"

    if retrieval_hit_at_5 and label_correct:
        return "pipeline_success"
    if retrieval_hit_at_5 and not label_correct:
        return "verifier_error_with_gold_retrieved"
    if not retrieval_hit_at_5 and not label_correct:
        return "retrieval_or_evidence_failure"
    return "correct_label_without_gold_hit"


def run_one_experiment(tokenizer, model, experiment):
    rows = []
    retrieval_method = experiment["retrieval_method"]
    retrieved_map = retrieved_maps[retrieval_method]
    model_tag = experiment["model_tag"]
    quantization = MODEL_CONFIGS[model_tag]["quantization"]

    for item in tqdm(dataset, desc=experiment["tag"]):
        retrieval_info = get_retrieval_info(item, retrieved_map)
        prompt = build_prompt(item, retrieved_map)
        raw_response = generate_one(tokenizer, model, prompt)
        predicted_label, predicted_explanation, parse_error = parse_verification_output(raw_response)

        gold_label = str(item["label"])
        label_correct = predicted_label == gold_label
        is_verifiable = gold_label in VERIFIABLE_LABELS
        fever_style_correct = is_verifiable and label_correct and retrieval_info["retrieval_hit_at_5"]

        row = {
            "experiment_tag": experiment["tag"],
            "retrieval_method": retrieval_method,
            "model_tag": model_tag,
            "quantization": quantization or "full_fp16",
            "claim_id": str(item["claim_id"]),
            "claim": item["claim"],
            "claim_type": item["claim_type"],
            "gold_label": gold_label,
            "predicted_label": predicted_label,
            "predicted_explanation": predicted_explanation,
            "gold_evidence_ids": retrieval_info["gold_evidence_ids"],
            "retrieved_chunk_ids": retrieval_info["retrieved_chunk_ids"],
            "retrieval_hit_at_5": retrieval_info["retrieval_hit_at_5"],
            "gold_rank": retrieval_info["gold_rank"],
            "is_verifiable": is_verifiable,
            "label_correct": label_correct,
            "fever_style_correct": fever_style_correct,
            "error_category": classify_end_to_end_result(
                gold_label,
                label_correct,
                retrieval_info["retrieval_hit_at_5"],
            ),
            "raw_response": raw_response,
            "parse_error": parse_error is not None,
            "parse_error_message": parse_error,
        }
        rows.append(row)

    return rows


In [8]:
all_predictions = []

# Group experiments by model so each model is loaded only once.
experiments_by_model = {}
for experiment in EXPERIMENTS:
    experiments_by_model.setdefault(experiment["model_tag"], []).append(experiment)

for model_tag, model_experiments in experiments_by_model.items():
    print("\n" + "=" * 80)
    print("Model:", model_tag)

    pending_experiments = []
    for experiment in model_experiments:
        out_path = OUTPUT_DIR / f"{RUN_TAG}_{experiment['tag']}_predictions.jsonl"

        if out_path.exists() and not FORCE_RERUN:
            print("Skip existing:", out_path)
            rows = read_jsonl(out_path)
            all_predictions.extend(rows)
        else:
            pending_experiments.append(experiment)

    if not pending_experiments:
        continue

    tokenizer = None
    model = None
    model_config = MODEL_CONFIGS[model_tag]

    try:
        tokenizer, model = load_llm(model_config["name"], model_config["quantization"])
        print("Device map:", getattr(model, "hf_device_map", None))

        for experiment in pending_experiments:
            out_path = OUTPUT_DIR / f"{RUN_TAG}_{experiment['tag']}_predictions.jsonl"
            rows = run_one_experiment(tokenizer, model, experiment)
            save_jsonl(rows, out_path)
            try:
                api.upload_file(
                    path_or_fileobj=str(out_path),
                    path_in_repo=out_path.name,
                    repo_id=HF_REPO_ID,
                    repo_type="dataset",
                    token=HF_TOKEN,
                )
                print("Uploaded to Hugging Face successfully:", out_path.name)
            except Exception as e:
                print("Upload failed:", e)
                
            print("Saved:", out_path)
            all_predictions.extend(rows)

    finally:
        # Explicit cleanup prevents the next model from inheriting occupied GPU memory.
        print("Cleaning up model:", model_tag)
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer

        model = None
        tokenizer = None
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

all_path = OUTPUT_DIR / f"{RUN_TAG}_end_to_end_predictions_all.jsonl"
save_jsonl(all_predictions, all_path)

print("Saved all:", all_path)
print("Total predictions:", len(all_predictions))



Model: qwen25_3b_full


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


hybrid_reranker_qwen25_7b_8bit:   0%|          | 0/1050 [00:00<?, ?it/s]

Uploaded to Hugging Face successfully: test_hybrid_reranker_qwen25_7b_8bit_predictions.jsonl
Saved: /kaggle/working/datasets/lanthanhvi/end_to_end_outputs/test_hybrid_reranker_qwen25_7b_8bit_predictions.jsonl
Cleaning up model: qwen25_7b_8bit
Saved all: /kaggle/working/datasets/lanthanhvi/end_to_end_outputs/test_end_to_end_predictions_all.jsonl
Total predictions: 5250


## Metric tổng hợp

- `accuracy` và `macro_f1`: tính trên toàn bộ ba nhãn.
- `evidence_recall_at_5`: tính trên `SUPPORTED` và `REFUTED`.
- `fever_style_score`: một claim khả kiểm chứng chỉ đúng khi nhãn đúng và gold evidence xuất hiện trong Top-5.


In [9]:
def compute_metric_row(group):
    y_true = group["gold_label"].astype(str).tolist()
    y_pred = group["predicted_label"].astype(str).tolist()
    per_label = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)

    verifiable_group = group[group["is_verifiable"].fillna(False)].copy()
    evidence_recall_at_5 = verifiable_group["retrieval_hit_at_5"].astype(bool).mean()
    fever_style_score = verifiable_group["fever_style_correct"].astype(bool).mean()

    return {
        "experiment_tag": group["experiment_tag"].iloc[0],
        "retrieval_method": group["retrieval_method"].iloc[0],
        "model_tag": group["model_tag"].iloc[0],
        "quantization": group["quantization"].iloc[0],
        "num_samples": len(group),
        "num_verifiable_samples": len(verifiable_group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "f1_supported": per_label[0],
        "f1_refuted": per_label[1],
        "f1_not_enough_evidence": per_label[2],
        "evidence_recall_at_5": evidence_recall_at_5,
        "fever_style_score": fever_style_score,
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
    }


df = pd.DataFrame(all_predictions)
if df.empty:
    raise ValueError("No predictions found. Run prediction cells first.")

summary_rows = []
for _, group in df.groupby("experiment_tag", sort=False):
    summary_rows.append(compute_metric_row(group))

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / f"{RUN_TAG}_end_to_end_summary.csv"
summary_df.to_csv(summary_path, index=False)

# These checks catch metric implementation errors immediately.
if not (summary_df["fever_style_score"] <= summary_df["evidence_recall_at_5"] + 1e-12).all():
    raise AssertionError("FEVER-style score cannot exceed Evidence Recall@5.")

summary_df


,experiment_tag,retrieval_method,model_tag,quantization,num_samples,num_verifiable_samples,accuracy,macro_f1,f1_supported,f1_refuted,f1_not_enough_evidence,evidence_recall_at_5,fever_style_score,parse_error_count
0,bm25_qwen25_3b_full,bm25,qwen25_3b_full,full_fp16,1050,700,0.341905,0.184621,0.044568,0.005698,0.503597,0.912857,0.012857,0
1,dense_qwen25_3b_full,dense,qwen25_3b_full,full_fp16,1050,700,0.340000,0.182401,0.044321,0.000000,0.502882,0.835714,0.011429,1
2,hybrid_qwen25_3b_full,hybrid,qwen25_3b_full,full_fp16,1050,700,0.340000,0.180660,0.039106,0.000000,0.502874,0.940000,0.010000,0
3,hybrid_reranker_qwen25_3b_full,hybrid_reranker,qwen25_3b_full,full_fp16,1050,700,0.348571,0.199785,0.086022,0.005698,0.507636,0.962857,0.024286,2
4,hybrid_reranker_qwen25_7b_8bit,hybrid_reranker,qwen25_7b_8bit,8bit,1050,700,0.618095,0.612310,0.688053,0.570481,0.578397,0.962857,0.677143,19


## Confusion matrix


In [10]:
cm_rows = []

for experiment_tag, group in df.groupby("experiment_tag", sort=False):
    cm = confusion_matrix(group["gold_label"], group["predicted_label"], labels=LABELS)

    for i, true_label in enumerate(LABELS):
        for j, predicted_label in enumerate(LABELS):
            cm_rows.append({
                "experiment_tag": experiment_tag,
                "retrieval_method": group["retrieval_method"].iloc[0],
                "model_tag": group["model_tag"].iloc[0],
                "true_label": true_label,
                "predicted_label": predicted_label,
                "count": int(cm[i, j]),
            })

cm_df = pd.DataFrame(cm_rows)
cm_path = OUTPUT_DIR / f"{RUN_TAG}_end_to_end_confusion_matrix.csv"
cm_df.to_csv(cm_path, index=False)
cm_df.head(30)


,experiment_tag,retrieval_method,model_tag,true_label,predicted_label,count
0,bm25_qwen25_3b_full,bm25,qwen25_3b_full,SUPPORTED,SUPPORTED,8
1,bm25_qwen25_3b_full,bm25,qwen25_3b_full,SUPPORTED,REFUTED,0
2,bm25_qwen25_3b_full,bm25,qwen25_3b_full,SUPPORTED,NOT_ENOUGH_EVIDENCE,342
3,bm25_qwen25_3b_full,bm25,qwen25_3b_full,REFUTED,SUPPORTED,1
4,bm25_qwen25_3b_full,bm25,qwen25_3b_full,REFUTED,REFUTED,1
5,bm25_qwen25_3b_full,bm25,qwen25_3b_full,REFUTED,NOT_ENOUGH_EVIDENCE,348
6,bm25_qwen25_3b_full,bm25,qwen25_3b_full,NOT_ENOUGH_EVIDENCE,SUPPORTED,0
7,bm25_qwen25_3b_full,bm25,qwen25_3b_full,NOT_ENOUGH_EVIDENCE,REFUTED,0
8,bm25_qwen25_3b_full,bm25,qwen25_3b_full,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,350
9,dense_qwen25_3b_full,dense,qwen25_3b_full,SUPPORTED,SUPPORTED,8


## Metric theo claim_type


In [11]:
claim_type_rows = []

for (experiment_tag, claim_type), group in df.groupby(["experiment_tag", "claim_type"], sort=False):
    y_true = group["gold_label"].astype(str).tolist()
    y_pred = group["predicted_label"].astype(str).tolist()
    verifiable_group = group[group["is_verifiable"].fillna(False)].copy()

    evidence_recall_at_5 = (
        verifiable_group["retrieval_hit_at_5"].astype(bool).mean()
        if not verifiable_group.empty else None
    )
    fever_style_score = (
        verifiable_group["fever_style_correct"].astype(bool).mean()
        if not verifiable_group.empty else None
    )

    claim_type_rows.append({
        "experiment_tag": experiment_tag,
        "retrieval_method": group["retrieval_method"].iloc[0],
        "model_tag": group["model_tag"].iloc[0],
        "claim_type": claim_type,
        "num_samples": len(group),
        "num_verifiable_samples": len(verifiable_group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "evidence_recall_at_5": evidence_recall_at_5,
        "fever_style_score": fever_style_score,
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
    })

claim_type_df = pd.DataFrame(claim_type_rows)
claim_type_path = OUTPUT_DIR / f"{RUN_TAG}_end_to_end_claim_type_metrics.csv"
claim_type_df.to_csv(claim_type_path, index=False)
claim_type_df.head(30)


,experiment_tag,retrieval_method,model_tag,claim_type,num_samples,num_verifiable_samples,accuracy,macro_f1,evidence_recall_at_5,fever_style_score,parse_error_count
0,bm25_qwen25_3b_full,bm25,qwen25_3b_full,TIME_EVENT,151,50,0.668874,0.267196,0.900000,0.000000,0
1,bm25_qwen25_3b_full,bm25,qwen25_3b_full,PERSON_ROLE,254,184,0.287402,0.165181,0.907609,0.016304,0
2,bm25_qwen25_3b_full,bm25,qwen25_3b_full,GENEALOGY_SUCCESSION,81,68,0.172840,0.110870,0.926471,0.014706,0
3,bm25_qwen25_3b_full,bm25,qwen25_3b_full,POLITICAL_ADMINISTRATIVE,159,97,0.415094,0.239930,0.876289,0.041237,0
4,bm25_qwen25_3b_full,bm25,qwen25_3b_full,MILITARY_DIPLOMACY,357,268,0.252101,0.138462,0.932836,0.003731,0
5,bm25_qwen25_3b_full,bm25,qwen25_3b_full,NATURAL_OMEN_DISASTER,48,33,0.312500,0.158730,0.878788,0.000000,0
6,dense_qwen25_3b_full,dense,qwen25_3b_full,TIME_EVENT,151,50,0.688742,0.370415,0.840000,0.060000,0
7,dense_qwen25_3b_full,dense,qwen25_3b_full,PERSON_ROLE,254,184,0.283465,0.158712,0.875000,0.010870,1
8,dense_qwen25_3b_full,dense,qwen25_3b_full,GENEALOGY_SUCCESSION,81,68,0.197531,0.143915,0.911765,0.044118,0
9,dense_qwen25_3b_full,dense,qwen25_3b_full,POLITICAL_ADMINISTRATIVE,159,97,0.389937,0.187029,0.721649,0.000000,0


## Phân tích bottleneck

Bốn nhóm chính cho `SUPPORTED` và `REFUTED`:

- `pipeline_success`: lấy đúng evidence và dự đoán đúng nhãn.
- `verifier_error_with_gold_retrieved`: đã lấy đúng evidence nhưng verifier dự đoán sai.
- `retrieval_or_evidence_failure`: không lấy được gold evidence và nhãn cũng sai.
- `correct_label_without_gold_hit`: nhãn đúng dù gold evidence không xuất hiện trong Top-5.


In [12]:
bottleneck_rows = []

for (experiment_tag, error_category), group in df.groupby(["experiment_tag", "error_category"], sort=False):
    bottleneck_rows.append({
        "experiment_tag": experiment_tag,
        "retrieval_method": group["retrieval_method"].iloc[0],
        "model_tag": group["model_tag"].iloc[0],
        "error_category": error_category,
        "num_samples": len(group),
        "percentage_within_experiment": len(group) / len(df[df["experiment_tag"] == experiment_tag]),
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
    })

bottleneck_df = pd.DataFrame(bottleneck_rows)
bottleneck_path = OUTPUT_DIR / f"{RUN_TAG}_end_to_end_bottleneck_summary.csv"
bottleneck_df.to_csv(bottleneck_path, index=False)
bottleneck_df


,experiment_tag,retrieval_method,model_tag,error_category,num_samples,percentage_within_experiment,parse_error_count
0,bm25_qwen25_3b_full,bm25,qwen25_3b_full,verifier_error_with_gold_retrieved,630,0.600000,0
1,bm25_qwen25_3b_full,bm25,qwen25_3b_full,nei_label_correct,350,0.333333,0
2,bm25_qwen25_3b_full,bm25,qwen25_3b_full,retrieval_or_evidence_failure,61,0.058095,0
3,bm25_qwen25_3b_full,bm25,qwen25_3b_full,pipeline_success,9,0.008571,0
4,dense_qwen25_3b_full,dense,qwen25_3b_full,verifier_error_with_gold_retrieved,577,0.549524,1
5,dense_qwen25_3b_full,dense,qwen25_3b_full,nei_label_correct,349,0.332381,0
6,dense_qwen25_3b_full,dense,qwen25_3b_full,retrieval_or_evidence_failure,115,0.109524,0
7,dense_qwen25_3b_full,dense,qwen25_3b_full,pipeline_success,8,0.007619,0
8,dense_qwen25_3b_full,dense,qwen25_3b_full,nei_label_incorrect,1,0.000952,0
9,hybrid_qwen25_3b_full,hybrid,qwen25_3b_full,verifier_error_with_gold_retrieved,651,0.620000,0


## File chẩn đoán theo từng claim


In [13]:
diagnostic_columns = [
    "experiment_tag", "retrieval_method", "model_tag", "claim_id", "claim_type",
    "gold_label", "predicted_label", "label_correct", "gold_evidence_ids",
    "retrieved_chunk_ids", "retrieval_hit_at_5", "gold_rank",
    "fever_style_correct", "error_category", "parse_error",
]

diagnostics_df = df[diagnostic_columns].copy()
diagnostics_path = OUTPUT_DIR / f"{RUN_TAG}_end_to_end_diagnostics.csv"
diagnostics_df.to_csv(diagnostics_path, index=False)
diagnostics_df.head(20)


,experiment_tag,retrieval_method,model_tag,claim_id,claim_type,gold_label,predicted_label,label_correct,gold_evidence_ids,retrieved_chunk_ids,retrieval_hit_at_5,gold_rank,fever_style_correct,error_category,parse_error
0,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000001,TIME_EVENT,SUPPORTED,NOT_ENOUGH_EVIDENCE,False,[019f17e0-fc5a-7cf2-ba14-b20bc5b39825],"[019f17e0-fddb-7bda-92df-7efc745cb789, 019f17e...",True,2.0,False,verifier_error_with_gold_retrieved,False
1,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000003,TIME_EVENT,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,True,[019f17e0-fc5a-7cf2-ba14-b20bc5b39825],"[019f17e2-ee32-7185-8ac4-f713623905c2, 019f17e...",False,NaN,False,nei_label_correct,False
2,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000004,PERSON_ROLE,SUPPORTED,NOT_ENOUGH_EVIDENCE,False,[019f17e0-fc85-73fc-b591-3981c10946ae],"[019f17e0-fc85-73fc-b591-3981c10946ae, 019f17e...",True,1.0,False,verifier_error_with_gold_retrieved,False
3,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000005,GENEALOGY_SUCCESSION,REFUTED,NOT_ENOUGH_EVIDENCE,False,[019f17e0-fc85-73fc-b591-3981c10946ae],"[019f17e0-fc85-73fc-b591-3981c10946ae, 019f17e...",True,1.0,False,verifier_error_with_gold_retrieved,False
4,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000006,TIME_EVENT,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,True,[019f17e0-fc85-73fc-b591-3981c10946ae],"[019f17e0-fc85-73fc-b591-3981c10946ae, 019f17e...",True,1.0,False,nei_label_correct,False
5,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000007,POLITICAL_ADMINISTRATIVE,SUPPORTED,NOT_ENOUGH_EVIDENCE,False,[019f17e0-fd57-7f7e-8051-5da239cf14fd],"[019f17e0-fd57-7f7e-8051-5da239cf14fd, 019f17e...",True,1.0,False,verifier_error_with_gold_retrieved,False
6,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000008,POLITICAL_ADMINISTRATIVE,REFUTED,NOT_ENOUGH_EVIDENCE,False,[019f17e0-fd57-7f7e-8051-5da239cf14fd],"[019f17e0-fefe-74be-b3cc-0e4cab13c8bd, 019f17e...",True,4.0,False,verifier_error_with_gold_retrieved,False
7,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000009,POLITICAL_ADMINISTRATIVE,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,True,[019f17e0-fd57-7f7e-8051-5da239cf14fd],"[019f17e0-fd57-7f7e-8051-5da239cf14fd, 019f17e...",True,1.0,False,nei_label_correct,False
8,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000010,POLITICAL_ADMINISTRATIVE,SUPPORTED,NOT_ENOUGH_EVIDENCE,False,[019f17e0-fefe-74be-b3cc-0e4cab13c8bd],"[019f17e0-fefe-74be-b3cc-0e4cab13c8bd, 019f17e...",True,1.0,False,verifier_error_with_gold_retrieved,False
9,bm25_qwen25_3b_full,bm25,qwen25_3b_full,claim_000011,MILITARY_DIPLOMACY,REFUTED,NOT_ENOUGH_EVIDENCE,False,[019f17e0-fefe-74be-b3cc-0e4cab13c8bd],"[019f17e1-8248-77d3-a75e-3e4347cfa532, 019f17e...",True,2.0,False,verifier_error_with_gold_retrieved,False


## Nén output


In [14]:
zip_base = Path(f"/kaggle/working/end_to_end_outputs_clean_{RUN_TAG}")
zip_path = shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR)

print("ZIP:", zip_path)
print("Files:")

for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)


ZIP: /kaggle/working/end_to_end_outputs_clean_test.zip
Files:
- test_bm25_qwen25_3b_full_predictions.jsonl
- test_dense_qwen25_3b_full_predictions.jsonl
- test_end_to_end_bottleneck_summary.csv
- test_end_to_end_claim_type_metrics.csv
- test_end_to_end_confusion_matrix.csv
- test_end_to_end_diagnostics.csv
- test_end_to_end_predictions_all.jsonl
- test_end_to_end_summary.csv
- test_hybrid_qwen25_3b_full_predictions.jsonl
- test_hybrid_reranker_qwen25_3b_full_predictions.jsonl
- test_hybrid_reranker_qwen25_7b_8bit_predictions.jsonl


In [15]:
# import shutil
# # Xóa sạch thư mục kết quả cũ
# if OUTPUT_DIR.exists():
#     shutil.rmtree(OUTPUT_DIR)
# # Tạo lại thư mục rỗng
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)